<a href="https://colab.research.google.com/github/Prem-del127/Music-Genre-Classification/blob/main/Copy_of_Project_11_%E2%80%93_Music_Genre_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "andradaolteanu/gtzan-dataset-music-genre-classification"
    )

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.
Path to dataset files: /kaggle/input/gtzan-dataset-music-genre-classification


In [ ]:
# ==========================
# INSTALL LIBRARIES
# ==========================
!pip install librosa tensorflow scikit-learn matplotlib -q

# ==========================
# IMPORT LIBRARIES
# ==========================
import os
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

# ==========================
# DATASET PATH
# ==========================
# Using the path from the previous cell, assuming 'genres_original' is in a 'Data' subdirectory
DATASET_PATH = os.path.join(path, "Data", "genres_original")

# ==========================
# FEATURE EXTRACTION
# ==========================
features = []
labels = []

for genre in os.listdir(DATASET_PATH):
    genre_path = os.path.join(DATASET_PATH, genre)
    if os.path.isdir(genre_path):
        for file in os.listdir(genre_path):
            if file.endswith(".wav"):
                file_path = os.path.join(genre_path, file)
                try:
                    y, sr = librosa.load(file_path, duration=30)
                    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
                    mfcc_scaled = np.mean(mfcc.T, axis=0)
                    features.append(mfcc_scaled)
                    labels.append(genre)
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

# ==========================
# CONVERT TO ARRAY
# ==========================
X = np.array(features)
y = np.array(labels)

print("Features Shape:", X.shape)
print("Labels Shape:", y.shape)

# ==========================
# LABEL ENCODING
# ==========================
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)

# ==========================
# TRAIN TEST SPLIT
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42
)

# ==========================
# RESHAPE FOR CNN
# ==========================
X_train = X_train.reshape(X_train.shape[0], 40, 1)
X_test = X_test.reshape(X_test.shape[0], 40, 1)

# ==========================
# CNN MODEL
# ==========================
model = Sequential()
model.add(
    Conv1D(
        64,
        kernel_size=3,
        activation='relu',
        input_shape=(40,1)
    )
)
model.add(MaxPooling1D(pool_size=2))
model.add(
    Conv1D(
        128,
        kernel_size=3,
        activation='relu'
    )
)
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))

# ==========================
# COMPILE
# ==========================
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================
# TRAIN MODEL
# ==========================
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)

# ==========================
# EVALUATION
# ==========================
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("\nAccuracy:", accuracy*100)

# ==========================
# SAVE MODEL
# ==========================
model.save("/content/music_genre_model.h5")
print("Model Saved!")

/tmp/ipykernel_3381/2377491200.py:39: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, duration=30)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Error processing file /kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original/jazz/jazz.00054.wav: 
Features Shape: (999, 40)
Labels Shape: (999,)
Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.2541 - loss: 2.2506 - val_accuracy: 0.4400 - val_loss: 1.6045
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3967 - loss: 1.6812 - val_accuracy: 0.4650 - val_loss: 1.4178
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4556 - loss: 1.5081 - val_accuracy: 0.5050 - val_loss: 1.3132
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5494 - loss: 1.2959 - val_accuracy: 0.5500 - val_loss: 1.2264
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5857 - loss: 1.1881 - val_accuracy: 0.5650 - val_loss: 1.2237
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6258 - loss: 1.0620 - val_accuracy: 0.6050 - val_loss: 1.0634
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6521 - loss: 0.9957 - val_accuracy: 0.6000 - val_loss: 1.0768
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6746 - loss: 0.8976 - val_accuracy: 0.5650 - val_loss: 1.1


Accuracy: 62.00000047683716
Model Saved!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving _Ye aayegi 💖✨...... Mention yours 🤌🏻❤️__Follow _ridewithharshh_ for more 👈🏼_._._my girl_ mine_ she always come_ hunter 350_ royal enfie(.mp3 to _Ye aayegi 💖✨...... Mention yours 🤌🏻❤️__Follow _ridewithharshh_ for more 👈🏼_._._my girl_ mine_ she always come_ hunter 350_ royal enfie(.mp3


In [ ]:
import librosa
import numpy as np
from tensorflow.keras.models import load_model

model = load_model('/content/music_genre_model.h5')

audio_file = list(uploaded.keys())[0]

y, sr = librosa.load(audio_file, duration=30)

mfcc = librosa.feature.mfcc(
    y=y,
    sr=sr,
    n_mfcc=40
)

mfcc = np.mean(mfcc.T, axis=0)
mfcc = mfcc.reshape(1,40,1)

prediction = model.predict(mfcc)

genres = [
    'blues',
    'classical',
    'country',
    'disco',
    'hiphop',
    'jazz',
    'metal',
    'pop',
    'reggae',
    'rock'
]

predicted_index = np.argmax(prediction)

print("Genre :", genres[predicted_index])
print("Confidence :", round(np.max(prediction)*100,2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
Genre : country
Confidence : 99.8 %


In [ ]:
!ls /content/

 music_genre_model.h5
 sample_data
'_Ye aayegi 💖✨...... Mention yours 🤌🏻❤️__Follow _ridewithharshh_ for more 👈🏼_._._my girl_ mine_ she always come_ hunter 350_ royal enfie(.mp3'
